<a href="https://colab.research.google.com/github/Amdjed117/twitter-airline-sentiment-analysis/blob/main/twitter_airline_sentiment_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [18]:
import os

base_dir = "TwitterAirlineSentiment"

folders = [
    "data",
    "models/distilbert_sentiment",
    "outputs"
]

for folder in folders:
    os.makedirs(os.path.join(base_dir, folder), exist_ok=True)

print("Project structure created!")


Project structure created!


In [19]:
!git clone https://github.com/ruchitgandhi/Twitter-Airline-Sentiment-Analysis.git

fatal: destination path 'Twitter-Airline-Sentiment-Analysis' already exists and is not an empty directory.


In [20]:

import pandas as pd

df = pd.read_csv("TwitterAirlineSentiment/data/Tweets.csv")
df.head()

,tweet_id,airline_sentiment,airline_sentiment_confidence,negativereason,negativereason_confidence,airline,airline_sentiment_gold,name,negativereason_gold,retweet_count,text,tweet_coord,tweet_created,tweet_location,user_timezone
0,570306133677760513,neutral,1.0000,NaN,NaN,Virgin America,NaN,cairdin,NaN,0,@VirginAmerica What @dhepburn said.,NaN,2015-02-24 11:35:52 -0800,NaN,Eastern Time (US & Canada)
1,570301130888122368,positive,0.3486,NaN,0.0000,Virgin America,NaN,jnardino,NaN,0,@VirginAmerica plus you've added commercials t...,NaN,2015-02-24 11:15:59 -0800,NaN,Pacific Time (US & Canada)
2,570301083672813571,neutral,0.6837,NaN,NaN,Virgin America,NaN,yvonnalynn,NaN,0,@VirginAmerica I didn't today... Must mean I n...,NaN,2015-02-24 11:15:48 -0800,Lets Play,Central Time (US & Canada)
3,570301031407624196,negative,1.0000,Bad Flight,0.7033,Virgin America,NaN,jnardino,NaN,0,@VirginAmerica it's really aggressive to blast...,NaN,2015-02-24 11:15:36 -0800,NaN,Pacific Time (US & Canada)
4,570300817074462722,negative,1.0000,Can't Tell,1.0000,Virgin America,NaN,jnardino,NaN,0,@VirginAmerica and it's a really big bad thing...,NaN,2015-02-24 11:14:45 -0800,NaN,Pacific Time (US & Canada)


In [21]:
df = df[df["airline_sentiment"].isin(["positive", "negative"])]
df = df[["text", "airline_sentiment"]]
df.rename(columns={"airline_sentiment": "label"}, inplace=True)

print(df["label"].value_counts())


label
negative    9178
positive    2363
Name: count, dtype: int64


In [22]:
df["label"] = df["label"].map({
    "negative": 0,
    "positive": 1
})


In [23]:
from sklearn.model_selection import train_test_split

train_df, temp_df = train_test_split(
    df,
    test_size=0.30,
    random_state=42,
    stratify=df["label"]
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=42,
    stratify=temp_df["label"]
)


In [24]:
train_df.to_csv("TwitterAirlineSentiment/data/train.csv", index=False)
val_df.to_csv("TwitterAirlineSentiment/data/val.csv", index=False)
test_df.to_csv("TwitterAirlineSentiment/data/test.csv", index=False)


In [25]:
print("Train:", len(train_df))
print("Val:", len(val_df))
print("Test:", len(test_df))


Train: 8078
Val: 1731
Test: 1732


In [26]:
!pip install -q transformers torch scikit-learn pandas tqdm


In [27]:
from transformers import DistilBertTokenizerFast

tokenizer = DistilBertTokenizerFast.from_pretrained("distilbert-base-uncased")

def tokenize(data):
    return tokenizer(
        data["text"].tolist(),
        truncation=True,
        padding=True,
        max_length=128,
        return_tensors="pt"
    )


In [28]:
import torch

class TweetDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {k: v[idx] for k, v in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)


In [29]:
train_enc = tokenize(train_df)
val_enc = tokenize(val_df)

train_dataset = TweetDataset(train_enc, train_df["label"].tolist())
val_dataset = TweetDataset(val_enc, val_df["label"].tolist())


In [30]:
from transformers import DistilBertForSequenceClassification

model = DistilBertForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=2
)


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [31]:
from torch.utils.data import DataLoader
from torch.optim import AdamW
from tqdm import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=16)

optimizer = AdamW(model.parameters(), lr=5e-5)
epochs = 3


In [32]:
model.train()

for epoch in range(epochs):
    loop = tqdm(train_loader)
    total_loss = 0

    for batch in loop:
        optimizer.zero_grad()

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )

        loss = outputs.loss
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        loop.set_description(f"Epoch {epoch+1}")
        loop.set_postfix(loss=loss.item())

    print(f"Epoch {epoch+1} Loss: {total_loss/len(train_loader):.4f}")


Epoch 1: 100%|██████████| 505/505 [42:11<00:00,  5.01s/it, loss=0.187]


Epoch 1 Loss: 0.1966


Epoch 2: 100%|██████████| 505/505 [41:22<00:00,  4.92s/it, loss=0.21]


Epoch 2 Loss: 0.0895


Epoch 3: 100%|██████████| 505/505 [42:41<00:00,  5.07s/it, loss=0.0165]

Epoch 3 Loss: 0.0448


In [33]:
model.save_pretrained("TwitterAirlineSentiment/models/distilbert_sentiment")
tokenizer.save_pretrained("TwitterAirlineSentiment/models/distilbert_sentiment")


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('TwitterAirlineSentiment/models/distilbert_sentiment/tokenizer_config.json',
 'TwitterAirlineSentiment/models/distilbert_sentiment/tokenizer.json')

In [34]:
from sklearn.metrics import classification_report

test_enc = tokenize(test_df)
test_dataset = TweetDataset(test_enc, test_df["label"].tolist())
test_loader = DataLoader(test_dataset, batch_size=16)

model.eval()
preds, labels = [], []

with torch.no_grad():
    for batch in test_loader:
        outputs = model(
            batch["input_ids"].to(device),
            attention_mask=batch["attention_mask"].to(device)
        )
        preds.extend(torch.argmax(outputs.logits, axis=1).cpu().numpy())
        labels.extend(batch["labels"].numpy())

print(classification_report(labels, preds, target_names=["negative", "positive"]))


              precision    recall  f1-score   support

    negative       0.96      0.95      0.96      1377
    positive       0.82      0.86      0.84       355

    accuracy                           0.93      1732
   macro avg       0.89      0.91      0.90      1732
weighted avg       0.93      0.93      0.93      1732



In [35]:
def predict(text):
    enc = tokenizer(text, return_tensors="pt", truncation=True, padding=True).to(device)
    out = model(**enc)
    label = torch.argmax(out.logits, axis=1).item()
    return "positive" if label == 1 else "negative"

tests = [
    "Worst airline ever, never flying again",
    "Amazing flight, friendly crew",
    "Flight delayed but staff tried their best",
    "Everything was okay, nothing special"
]

for t in tests:
    print(t, "->", predict(t))


Worst airline ever, never flying again -> negative
Amazing flight, friendly crew -> positive
Flight delayed but staff tried their best -> negative
Everything was okay, nothing special -> negative


In [36]:
predictions = [predict(t) for t in test_df["text"].sample(50)]

pd.DataFrame({
    "text": test_df["text"].sample(50),
    "prediction": predictions
}).to_csv("TwitterAirlineSentiment/outputs/predictions.csv", index=False)


In [37]:
with open("TwitterAirlineSentiment/outputs/training_logs.txt", "w") as f:
    f.write("Model: DistilBERT\n")
    f.write("Epochs: 3\n")
    f.write("Optimizer: AdamW\n")


In [38]:
!zip -r TwitterAirlineSentiment.zip TwitterAirlineSentiment

  adding: TwitterAirlineSentiment/ (stored 0%)
  adding: TwitterAirlineSentiment/outputs/ (stored 0%)
  adding: TwitterAirlineSentiment/outputs/predictions.csv (deflated 52%)
  adding: TwitterAirlineSentiment/outputs/training_logs.txt (stored 0%)
  adding: TwitterAirlineSentiment/data/ (stored 0%)
  adding: TwitterAirlineSentiment/data/Tweets.csv (deflated 67%)
  adding: TwitterAirlineSentiment/data/test.csv (deflated 61%)
  adding: TwitterAirlineSentiment/data/val.csv (deflated 61%)
  adding: TwitterAirlineSentiment/data/train.csv (deflated 62%)
  adding: TwitterAirlineSentiment/models/ (stored 0%)
  adding: TwitterAirlineSentiment/models/distilbert_sentiment/ (stored 0%)
  adding: TwitterAirlineSentiment/models/distilbert_sentiment/tokenizer.json (deflated 71%)
  adding: TwitterAirlineSentiment/models/distilbert_sentiment/model.safetensors (deflated 8%)
  adding: TwitterAirlineSentiment/models/distilbert_sentiment/tokenizer_config.json (deflated 42%)
  adding: TwitterAirlineSentiment

In [39]:
!ls

sample_data		 Twitter-Airline-Sentiment-Analysis
TwitterAirlineSentiment  TwitterAirlineSentiment.zip


In [40]:
from google.colab import files
files.download("TwitterAirlineSentiment.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [41]:
!rm -rf TwitterAirlineSentiment/models

In [44]:
!zip -r TwitterAirlineSentiment.zip TwitterAirlineSentiment

  adding: TwitterAirlineSentiment/ (stored 0%)
  adding: TwitterAirlineSentiment/outputs/ (stored 0%)
  adding: TwitterAirlineSentiment/outputs/predictions.csv (deflated 52%)
  adding: TwitterAirlineSentiment/outputs/training_logs.txt (stored 0%)
  adding: TwitterAirlineSentiment/data/ (stored 0%)
  adding: TwitterAirlineSentiment/data/Tweets.csv (deflated 67%)
  adding: TwitterAirlineSentiment/data/test.csv (deflated 61%)
  adding: TwitterAirlineSentiment/data/val.csv (deflated 61%)
  adding: TwitterAirlineSentiment/data/train.csv (deflated 62%)


In [45]:
from google.colab import files
files.download("TwitterAirlineSentiment.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>